# GPU gate (M03.1 + M05.1): ModernBERT check + Qwen3.5-0.8B smoke

Run this notebook top-to-bottom on **Google Colab** (`Runtime -> Run all`, free T4 is enough,
~15-25 min including one-time downloads). It performs the two GPU-gated checks from
STATE.md without doing any real training:

1. **M03.1 - detector gate:** verify `answerdotai/ModernBERT-base` loads as a
   token-classification model with its OWN tokenizer (mBERT arrays are never reused),
   that the repo's BIO alignment round-trips on real dataset rows, that one real
   optimizer step runs, and how much memory it costs.
2. **M05.1 - generator gate:** verify `Qwen/Qwen3.5-0.8B` loads on a text-only path,
   follows the request schema's output contract on a pool-compliant prompt, and that a
   real LoRA adapter update (forward + backward + step) runs.

Everything (checkpoints, adapter, reports, dataset cache) is written to
**Google Drive** at `MyDrive/pii-redteam-rl/`, so nothing is lost when the VM is recycled.
The dataset locks are rebuilt from the pinned revision and asserted against the sha256
recorded in `docs/DATA_CARD.md` - a mismatch stops the run.

This is a measurement gate, not training: no scientific result is produced here.

In [ ]:
import datetime
import json
import os
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
RUN_ID = datetime.datetime.now(datetime.UTC).strftime("run-%Y%m%d-%H%M%S")
WORK = os.path.join(DRIVE_ROOT, RUN_ID)  # this run's artifacts
CACHE = os.path.join(DRIVE_ROOT, "cache")  # reusable dataset downloads

# Lock recorded in docs/DATA_CARD.md; update here if the lock ever changes.
EXPECTED_FINAL_EVAL_SHA = "513335badf5752a340698d3c7efa7029a8780c51fde2675c80ef4208ce27d39c"

# Mount Google Drive FIRST: all outputs below are saved there.
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

REPORT = {"run_id": RUN_ID, "checks": {}, "notes": []}


def check(name, ok, detail=""):
    """Record and print one gate check. FAIL never crashes the notebook."""
    REPORT["checks"][name] = {"ok": bool(ok), "detail": str(detail)}
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" -- {detail}" if detail else ""))


print(f"run_id: {RUN_ID}")
print(f"drive workdir: {WORK}")

In [ ]:
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()

# Colab preinstalls torch; only pull the model stack (qwen3_5 needs a recent transformers).
# torchao must be upgraded too: Colab's preinstalled 0.10.0 breaks peft/transformers imports.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers",
        "accelerate",
        "peft",
        "torchao",
    ],
    check=True,
)
import accelerate  # noqa: E402
import peft  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from pii_redteam.detector import (  # noqa: E402
    bio_labels_from_spans,
    spans_from_bio_labels,
    trim_span_whitespace,
)
from pii_redteam.data import QuarantinedRecord, to_canonical  # noqa: E402
from pii_redteam.requests import (  # noqa: E402
    is_requestable_name,
    load_person_name_pool,
    render_prompt,
    seeded_sampler,
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    if torch.cuda.is_available()
    else 0
)
REPORT["environment"] = {
    "repo_commit": REPO_COMMIT,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "gpu": gpu_name,
    "vram_gb": vram_gb,
}
print(json.dumps(REPORT["environment"], indent=2))

In [ ]:
# Rebuild the locked split artifacts from the pinned revision (downloads ~130 MB once,
# cached on Drive), then assert the final-eval partition matches the recorded lock.
# A different hash means the carve is not the locked one -> STOP, do not train on it.
import shutil

splits_dir = os.path.join(REPO_DIR, "artifacts", "splits")
try:
    subprocess.run(
        [sys.executable, "scripts/carve_splits.py", "--cache-dir", CACHE, "--out", splits_dir],
        cwd=REPO_DIR,
        env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")),
        check=True,
    )
    manifest = json.load(open(os.path.join(splits_dir, "final_eval_manifest.json")))
    ok = manifest["final_eval_sha256"] == EXPECTED_FINAL_EVAL_SHA
    check(
        "final-eval lock reproduces on Colab",
        ok,
        manifest["final_eval_sha256"]
        if ok
        else "expected "
        + EXPECTED_FINAL_EVAL_SHA
        + ", got "
        + manifest["final_eval_sha256"]
        + " - STOP: do not train or author prompts from a carve that does not match the lock",
    )
    REPORT["locks"] = {
        "final_eval_sha256": manifest["final_eval_sha256"],
        "final_eval_n": manifest["final_eval_n"],
        "seed": manifest["seed"],
    }
except Exception as e:  # noqa: BLE001
    check("final-eval lock reproduces on Colab", False, repr(e))

if not REPORT["checks"].get("final-eval lock reproduces on Colab", {}).get("ok"):
    raise SystemExit("Lock mismatch - refusing to continue (see message above).")

# Persist the lock artifacts of this run to Drive.
for fname in ("final_eval_manifest.json", "name_pools.json", "report.json"):
    src = os.path.join(splits_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(WORK, fname))

FINAL_EVAL_IDS = {r["example_id"] for r in manifest["rows"]}
TRAIN_FILE = os.path.join(CACHE, "1english_openpii_30k.jsonl")


def load_train_rows(n):
    """First n usable train-side rows (quarantine-filtered, final-eval rows excluded)."""
    rows = []
    with open(TRAIN_FILE, encoding="utf-8") as f:
        for line in f:
            if len(rows) >= n:
                break
            try:
                canonical = to_canonical(json.loads(line))
            except QuarantinedRecord:
                continue
            if canonical["example_id"] in FINAL_EVAL_IDS or not canonical["spans"]:
                continue
            rows.append(canonical)
    return rows


ROWS = load_train_rows(50)
print(f"loaded {len(ROWS)} train-side rows with PERSON spans for the checks below")

In [ ]:
# ---- M03.1: ModernBERT detector gate ----------------------------------------
import time

from transformers import AutoModelForTokenClassification, AutoTokenizer

MB_ID = "answerdotai/ModernBERT-base"
LABEL2ID = {"O": 0, "B-PERSON": 1, "I-PERSON": 2}

mb_tok = AutoTokenizer.from_pretrained(MB_ID)
mb_model = AutoModelForTokenClassification.from_pretrained(
    MB_ID,
    num_labels=len(LABEL2ID),
    id2label={i: l for l, i in LABEL2ID.items()},
    label2id=LABEL2ID,
).cuda()
check("ModernBERT loads for token classification", True, MB_ID)

# 1) Round-trip: real rows -> tokenizer offsets -> BIO labels -> decoded spans.
# BPE tokenizers prefix a word's first token with the preceding space, so recovered
# spans are trimmed before comparison (verified on CPU: 29/50 -> 50/50 exact).
exact = mismatched = truncated = 0
mismatches = []
for row in ROWS:
    enc = mb_tok(row["text"], return_offsets_mapping=True, truncation=True, max_length=512)
    if len(enc["input_ids"]) >= 512:
        truncated += 1
        continue
    offsets = enc["offset_mapping"]
    labels = bio_labels_from_spans(offsets, row["spans"])
    recovered = trim_span_whitespace(row["text"], spans_from_bio_labels(offsets, labels))
    if recovered == row["spans"]:
        exact += 1
    else:
        mismatched += 1
        mismatches.append({"example_id": row["example_id"], "orig": row["spans"], "got": recovered})
check(
    "BIO round-trip on real rows",
    mismatched == 0 and exact >= 20,
    f"exact={exact} mismatched={mismatched} skipped_as_too_long={truncated}",
)
if mismatches:
    print(json.dumps(mismatches[:3], indent=2, ensure_ascii=False))

# 2) One REAL optimizer step on a tiny batch (fresh random head, plumbing only).
torch.cuda.reset_peak_memory_stats()
mb_model.train()
batch = ROWS[:4]
enc = mb_tok(
    [r["text"] for r in batch],
    return_offsets_mapping=True,
    truncation=True,
    max_length=128,
    padding=True,
)
labels = []
for offsets, row in zip(enc["offset_mapping"], batch):
    seq = []
    for (s, e), lab in zip(offsets, bio_labels_from_spans(offsets, row["spans"])):
        seq.append(-100 if (s, e) == (0, 0) else LABEL2ID[lab])
    labels.append(seq)
input_ids = torch.tensor(enc["input_ids"]).cuda()
attention = torch.tensor(enc["attention_mask"]).cuda()
labels_t = torch.tensor(labels).cuda()
t0 = time.time()
out = mb_model(input_ids=input_ids, attention_mask=attention, labels=labels_t)
out.loss.backward()
grad_sum = sum(p.grad.abs().sum().item() for p in mb_model.parameters() if p.grad is not None)
torch.optim.AdamW(mb_model.parameters(), lr=5e-5).step()
step_s = time.time() - t0
peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
check(
    "ModernBERT real training step",
    torch.isfinite(out.loss) and grad_sum > 0,
    f"loss={out.loss.item():.4f} grad_sum={grad_sum:.3e} step={step_s:.2f}s peak_mem={peak_gb} GB",
)
mb_model.save_pretrained(os.path.join(WORK, "modernbert-smoke"))
mb_tok.save_pretrained(os.path.join(WORK, "modernbert-smoke"))
REPORT["modernbert"] = {
    "model_id": MB_ID,
    "roundtrip_exact": exact,
    "roundtrip_mismatched": mismatched,
    "train_step_loss": out.loss.item(),
    "grad_sum": grad_sum,
    "step_seconds": step_s,
    "peak_mem_gb": peak_gb,
}
print(f"checkpoint saved to {WORK}/modernbert-smoke")

In [ ]:
# ---- M05.1: Qwen3.5-0.8B generator gate --------------------------------------
import time

from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

QWEN_ID = "Qwen/Qwen3.5-0.8B"

# Request drawn ONLY from the train-side pool (R06), through the repo's sampler.
# The loader already drops R07 pseudo values; sample from the requestable subset.
name_pools = json.load(open(os.path.join(splits_dir, "name_pools.json")))
allowed, pool_prov = load_person_name_pool(os.path.join(splits_dir, "name_pools.json"))
pool_names = [
    e["value"]
    for e in name_pools["train_side_pool"]
    if e["value"] in allowed and is_requestable_name(e["value"])
]
requests = seeded_sampler(pool_names[:8], seed=REPORT["locks"]["seed"], allowed_names=allowed)
request = requests[0]
prompt = render_prompt(request)
target_name = request["task"]["entities"][0]["value"]
print("--- prompt ---")
print(prompt)
REPORT["qwen_request"] = {"request": request, "pool_provenance": pool_prov}

# Text-only load. Qwen3.5 is hybrid (vision encoder); try the text-only class first,
# fall back to the conditional-generation class used with text-only inputs.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
load_path = ""
try:
    qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_ID, dtype=dtype, device_map="cuda")
    load_path = "AutoModelForCausalLM"
except Exception as e:  # noqa: BLE001
    print(f"AutoModelForCausalLM failed ({e!r}); falling back to the full model, text-only inputs")
    qwen_model = AutoModelForImageTextToText.from_pretrained(
        QWEN_ID, dtype=dtype, device_map="cuda"
    )
    load_path = "AutoModelForImageTextToText (text-only inputs)"
qwen_tok = AutoTokenizer.from_pretrained(QWEN_ID)
check("Qwen3.5-0.8B text-only load", True, f"{load_path} dtype={dtype}")

# Non-thinking mode explicitly where the template supports it (0.8B defaults to it).
messages = [{"role": "user", "content": prompt}]
chat_text = ""
try:
    chat_text = qwen_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    thinking = "enable_thinking=False"
except TypeError:
    chat_text = qwen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    thinking = "template default (no enable_thinking kwarg)"
except Exception as e:  # noqa: BLE001
    thinking = f"FAILED: {e!r}"
check("chat template renders (non-thinking)", bool(chat_text), thinking)

torch.cuda.reset_peak_memory_stats()
enc = qwen_tok(chat_text, return_tensors="pt").to("cuda")
t0 = time.time()
gen = qwen_model.generate(**enc, max_new_tokens=120, do_sample=False)
gen_s = time.time() - t0
generated = qwen_tok.decode(gen[0][enc["input_ids"].shape[1] :], skip_special_tokens=True)
print("--- generated ---")
print(generated)

# Output contract: JSON object with a "text" field, name exactly once.
try:
    parsed = json.loads(generated)
    gen_text = parsed.get("text") if isinstance(parsed, dict) else None
    json_ok = isinstance(gen_text, str)
except json.JSONDecodeError:
    gen_text, json_ok = None, False
occurrences = gen_text.count(target_name) if gen_text else -1
check(
    "output contract (JSON text field, name exactly once)",
    json_ok and occurrences == 1,
    f"json_ok={json_ok} occurrences_of_name={occurrences}",
)
qwen_peak = round(torch.cuda.max_memory_allocated() / 1e9, 2)

# One REAL LoRA adapter update on the generated pair (plumbing, not SFT).
lora_ok, lora_detail = False, "not attempted"
try:
    from peft import LoraConfig, get_peft_model

    linear_names = [
        n.split(".")[-1] for n, m in qwen_model.named_modules() if isinstance(m, torch.nn.Linear)
    ]
    targets = sorted(
        {n for n in linear_names if n in {"q_proj", "k_proj", "v_proj", "o_proj"}}
    ) or ["q_proj"]
    qwen_model = get_peft_model(
        qwen_model,
        LoraConfig(
            r=8, lora_alpha=16, target_modules=targets, lora_dropout=0.0, task_type="CAUSAL_LM"
        ),
    )
    qwen_model.train()
    seq = qwen_tok(chat_text + (gen_text or generated), return_tensors="pt").to("cuda")
    prompt_len = enc["input_ids"].shape[1]
    labels = seq["input_ids"].clone()
    labels[:, :prompt_len] = -100
    loss = qwen_model(**seq, labels=labels).loss
    loss.backward()
    trainable = [p for p in qwen_model.parameters() if p.requires_grad]
    grad_sum = sum(p.grad.abs().sum().item() for p in trainable if p.grad is not None)
    torch.optim.AdamW(trainable, lr=1e-4).step()
    lora_ok = bool(loss.isfinite()) and grad_sum > 0
    lora_detail = f"targets={targets} loss={loss.item():.4f} grad_sum={grad_sum:.3e}"
    qwen_model.save_pretrained(os.path.join(WORK, "qwen-lora-smoke"))
except Exception as e:  # noqa: BLE001
    lora_detail = f"{type(e).__name__}: {e}"
check("Qwen LoRA adapter update", lora_ok, lora_detail)

REPORT["qwen"] = {
    "model_id": QWEN_ID,
    "load_path": load_path,
    "dtype": str(dtype),
    "thinking_mode": thinking,
    "generation_seconds": round(gen_s, 2),
    "json_ok": json_ok,
    "name_occurrences": occurrences,
    "lora_ok": lora_ok,
    "lora_detail": lora_detail,
    "peak_mem_gb": qwen_peak,
    "generated_raw": generated,
}
print(f"adapter + report artifacts under {WORK}")

Final cell: aggregate report, saved to Drive. Copy the `gpu_gate_report.json` path and the
printed summary back into STATE.md as the evidence for the M03.1/M05.1 gate.

In [ ]:
REPORT["finished_utc"] = datetime.datetime.now(datetime.UTC).isoformat()

report_path = os.path.join(WORK, "gpu_gate_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(REPORT, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("GPU GATE SUMMARY")
print("=" * 70)
for name, res in REPORT["checks"].items():
    print(f"[{'PASS' if res['ok'] else 'FAIL'}] {name}")
    if res["detail"]:
        print(f"       {res['detail']}")
print("=" * 70)
print(f"report:      {report_path}")
print(f"checkpoints: {WORK}/modernbert-smoke, {WORK}/qwen-lora-smoke (if saved)")
print(f"dataset cache (reused next run): {CACHE}")
print("Next: copy the summary + report path into STATE.md as M03.1/M05.1 evidence.")
if not all(res["ok"] for res in REPORT["checks"].values()):
    print("NOTE: some checks FAILED - record them honestly in STATE.md; do not retune silently.")